# PART 3

# Notebook Imports
 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, HalvingGridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

import pprint

%matplotlib inline

# Constants

In [ ]:
DATASET_PATH = "dataset.csv"
RANDOM_STATE = 1
CROSS_VALIDATION_SETS = 5
LABEL_NAMES = ["English", "Urdu", "Mixed"]

In [ ]:
def plot(x, y, xlabel, ylabel, title):
    plt.plot(x,y)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(x)
    plt.show()

# Importing and Splitting Data

In [ ]:
data = pd.read_csv(DATASET_PATH)
data

In [ ]:
# randomly shuffle dataframe

train = data.sample(frac=0.8, random_state=RANDOM_STATE)
test = data.drop(train.index)

print("Length of train dataset:", len(train))
print("Length of test dataset:", len(test))

In [ ]:
x_train = np.array(train)[:, 1:-1]
y_train = np.array(train)[:, -1]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
x_test = np.array(test)[:, 1:-1]
y_test = np.array(test)[:, -1]

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
scaler = StandardScaler()
scaler = scaler.fit(x_train)

x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

# Training And Validation

In [ ]:
# hyper parameters

LEARNING_RATE_INIT = 0.0001
ALPHA = 0.012
HIDDEN_LAYERS = (1024, 512, 512, 256, 32)

In [ ]:
model = MLPClassifier(hidden_layer_sizes=(HIDDEN_LAYERS), alpha=ALPHA, learning_rate_init=LEARNING_RATE_INIT)
model.fit(x_train, y_train)

In [ ]:
loss = model.loss_curve_

plt.style.use("ggplot")
plt.figure(figsize=(10, 10))
plt.plot(range(len(loss)), loss)
plt.xlabel("Iterations")
plt.ylabel("Cost")
plt.title("Cost against iterations for best Estimator")
plt.show()

# 5 Fold CV

In [ ]:
nn = MLPClassifier(hidden_layer_sizes=(HIDDEN_LAYERS), alpha=ALPHA, learning_rate_init=LEARNING_RATE_INIT)
nn.fit(x_train, y_train)

In [ ]:
cv_scores = cross_val_score(nn, x_train, y_train, cv=CROSS_VALIDATION_SETS)

In [ ]:
x = [x for x in range(1, CROSS_VALIDATION_SETS + 1)]

plot(x, cv_scores, 'CV Sets', 'Accuracy', '5-Fold CV Accuracy Plot')

# Testing and Evaluation

In [ ]:
preds = model.predict(x_test)

In [ ]:
print(classification_report(y_test, preds, target_names=LABEL_NAMES))

In [ ]:
conf = confusion_matrix(y_test, preds)
matrix_plotter = ConfusionMatrixDisplay(conf, display_labels=LABEL_NAMES)


a = matrix_plotter.plot(ax=None)
a.ax_.grid()
plt.show()